# Testing the model

Using your solution so far, test the model on new data.

The new data is located in the ‘Bank_data_testing.csv’.

Good luck!

## Import the relevant libraries

In [1]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import matplotlib.pyplot as plt
import seaborn as sns
sns.set()

#Apply a fix to the statsmodels library
from scipy import stats
stats.chisqprob = lambda chisq, df: stats.chi2.sf(chisq, df)

## Load the data

Load the ‘Bank_data.csv’ dataset.

In [ ]:
raw_data_train = pd.read_csv('Bank-data.csv')

In [ ]:
raw_data_train

In [ ]:
raw_data_train.describe(include='all')

In [ ]:
data = raw_data_train.copy()
data = data.drop(['Unnamed: 0'], axis = 1)
data['y'] = data['y'].map({'yes':1, 'no':0})
data

### Declare the dependent and independent variables

Use 'duration' as the independet variable.

In [232]:
y = data['y']
X1 = data['duration']

### Simple Logistic Regression

Run the regression and graph the scatter plot.

In [ ]:
X = sm.add_constant(X1)
reg_log = sm.Logit(y, X)
results_log = reg_log.fit()
# Get the regression summary
results_log.summary()

In [ ]:
plt.scatter(X1, y, color = 'C0')
plt.xlabel('Duration', fontsize = 20)
plt.ylabel('Subscription', fontsize = 20)
plt.show()

## Expand the model

We can be omitting many causal factors in our simple logistic model, so we instead switch to a multivariate logistic regression model. Add the ‘interest_rate’, ‘march’, ‘credit’ and ‘previous’ estimators to our model and run the regression again.

### Declare the independent variable(s)

In [235]:
estimators = ['interest_rate','credit', 'march', 'previous','duration']

In [236]:
X1_extended = data[estimators]
y = data['y']

In [ ]:
X1_extended = sm.add_constant(X1_extended)
reg_logit = sm.Logit(y, X1_extended)
results_logit = reg_logit.fit()
# Get the regression summary
results_logit.summary()

### Confusion Matrix

Find the confusion matrix of the model and estimate its accuracy.

<i> For convenience we have already provided you with a function that finds the confusion matrix and the model accuracy.</i>

In [238]:
def confusion_matrix(predictors, actual_values, model):

        # Confusion matrix

        # Parameters
        # ----------
        # data: data frame or array
            # data is a data frame formatted in the same way as your input data (without the actual values)
            # e.g. const, var1, var2, etc. Order is very important!
        # actual_values: data frame or array
            # These are the actual values from the test_data
            # In the case of a logistic regression, it should be a single column with 0s and 1s

        # model: a LogitResults object
            # this is the variable where you have the fitted model
            # e.g. results_log in this course
        # ----------

        #Predict the values using the Logit model
        predicted_values = model.predict(predictors)
        # Specify the bins
        bins=np.array([0,0.5,1])
        # Create a histogram, where if values are between 0 and 0.5 tell will be considered 0
        # if they are between 0.5 and 1, they will be considered 1
        cm = np.histogram2d(actual_values, predicted_values, bins=bins)[0]
        # Calculate the accuracy
        accuracy = (cm[0,0]+cm[1,1])/cm.sum()
        # Return the confusion matrix and
        return cm, accuracy

In [ ]:
c_matrix_train = confusion_matrix(X1_extended, y, results_logit)
c_matrix_train

## Test the model

Load the test data from the ‘Bank_data_testing.csv’ file provided. (Remember to convert the outcome variable ‘y’ into Boolean).

### Load new data

In [240]:
raw_data_test = pd.read_csv('Bank-data-testing.csv')

In [ ]:
data_test = raw_data_test.copy()
data_test = data_test.drop(['Unnamed: 0'], axis = 1)
data_test['y'] = data_test['y'].map({'yes':1, 'no':0})
data_test

### Declare the dependent and the independent variables

In [242]:
y_test = data_test['y']
X1_test = data_test[estimators]
X_test = sm.add_constant(X1_test)

Determine the test confusion matrix and the test accuracy and compare them with the train confusion matrix and the train accuracy.

In [ ]:
# Test Accuracy
c_matrix_test = confusion_matrix(X_test, y_test, results_logit)
c_matrix_test

In [ ]:
# Test Accuracy - Summarized in a table (Confusion Matrix)
cm_table_test = pd.DataFrame(c_matrix_test[0])
cm_table_test = cm_table_test.rename(index={0:'Actual value is 0', 1:'Actual value is 1'},
                                                 columns={0:'Predicted 0', 1:'Predicted 1'})
cm_table_test

In [ ]:
# Train Accuracy
c_matrix_train = confusion_matrix(X1_extended, y, results_logit)
c_matrix_train

In [ ]:
# Train Accuracy - Summarized in a table (Confusion Matrix)
cm_table_train = pd.DataFrame(c_matrix_train[0])
cm_table_train = cm_table_train.rename(index={0:'Actual value is 0', 1:'Actual value is 1'},
                                                 columns={0:'Predicted 0', 1:'Predicted 1'})
cm_table_train

### Comparing the Train and Test Accuracies - Comments:
Looking at the test acccuracy we see a number which is a tiny but lower: 86.04%, compared to 86.29% for train accuracy.

In general, we always expect the test accuracy to be lower than the train one. If the test accuracy is higher, this is just due to luck.

Note that when you run the regression, you may get different numbers than us!